# TREA on Wikidata12K Cross-Lingual

**TREA:** Xu, Su, Xiong, Lehmann (WWW 2022) — *Time-aware Entity Alignment using Temporal Relational Attention*.

PyTorch reconstruction of TREA for **Wikidata12K cross-lingual** pairs:
`en_fr`, `en_fa`, `en_zh`, `en_ja`.

### Dataset layout
```
wikidata12k_crosslingual.zip
├── en_fr/  en_fa/  en_zh/  en_ja/
    ├── alignment_all.tsv
    ├── quadruples_en.tsv     # head_id rel_id tail_id begin end
    ├── quadruples_<lang>.tsv
    ├── entity2id_*.tsv
    └── relation2id.tsv
```

Timestamps are year-like strings (`1926-##-##`), converted to integer years.

**Caveat:** This is a reconstruction from the paper (not official code). Interpret results cautiously.

### Colab setup
1. Runtime → **GPU**
2. Put `wikidata12k_crosslingual.zip` on Drive or `/content`
3. Set `LANG_PAIR`
4. Run top to bottom

In [1]:
# ============================================================
# 1. Imports
# ============================================================

import os
import re
import gc
import zipfile
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 200
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Device: cuda
GPU: Tesla T4


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ============================================================
# 3. Configuration
# ============================================================

LANG_PAIR = "en_ja"  # en_fr | en_fa | en_zh | en_ja

ZIP_CANDIDATES = [
    Path("/content/drive/MyDrive/wikidata12k_crosslingual.zip"),
    Path("/content/wikidata12k_crosslingual.zip"),
]
EXTRACT_ROOT = Path("/content/wikidata12k_crosslingual")

# TREA hyperparameters (paper-like)
TRAIN_SEED = 200
DEPTH = 2
GAMMA = 0.0
LAMBDA_T = 0.001
LR = 0.005
DIM = 100
EPOCHS = 100

TOP_K = (1, 5, 10)
USE_CSLS = True
CSLS_K = 10

print("LANG_PAIR:", LANG_PAIR)
print("DEVICE:", DEVICE)

LANG_PAIR: en_ja
DEVICE: cuda


In [4]:
# ============================================================
# 4. Extract + locate pair
# ============================================================

zip_path = next((p for p in ZIP_CANDIDATES if p.exists()), None)
if zip_path is None:
    found = list(Path("/content").rglob("wikidata12k_crosslingual.zip"))
    zip_path = found[0] if found else None
if zip_path is None:
    raise FileNotFoundError("wikidata12k_crosslingual.zip not found")

print("Zip:", zip_path)
if not (EXTRACT_ROOT / ".extracted").exists():
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/content")
    (EXTRACT_ROOT / ".extracted").write_text("ok")

# folder may be /content/wikidata12k_crosslingual/en_fr
cands = [p.parent for p in Path("/content").rglob("alignment_all.tsv")
         if LANG_PAIR in str(p.parent) and p.parent.name == LANG_PAIR]
if not cands:
    cands = [p.parent for p in Path("/content").rglob("alignment_all.tsv")
             if LANG_PAIR in str(p.parent)]
if not cands:
    avail = sorted({p.parent.name for p in Path("/content").rglob("alignment_all.tsv")})
    raise FileNotFoundError(f"{LANG_PAIR} not found. Available: {avail}")

PAIR_DIR = cands[0]
lang_a, lang_b = LANG_PAIR.split("_")
print("PAIR_DIR:", PAIR_DIR)
for p in sorted(PAIR_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.name:24s} {p.stat().st_size/1024:.1f} KB")

Zip: /content/drive/MyDrive/wikidata12k_crosslingual.zip
PAIR_DIR: /content/wikidata12k_crosslingual/en_ja
  alignment_all.tsv        561.0 KB
  entity2id_en.tsv         417.0 KB
  entity2id_ja.tsv         359.9 KB
  quadruples_en.tsv        1412.4 KB
  quadruples_ja.tsv        888.4 KB
  relation2id.tsv          0.6 KB


In [5]:
# ============================================================
# 5. Load Wikidata12K quadruples + alignment
# ============================================================

def parse_time(tok: str) -> int:
    m = re.match(r"^(-?\d+)", tok.strip())
    return int(m.group(1)) if m else 0


def load_quadruples(path):
    triples, ents, rels, times = [], set(), set(), set()
    with open(path, "r", encoding="utf-8") as f:
        next(f)  # header
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 5:
                parts = line.split()
            if len(parts) < 5:
                continue
            h, r, t = int(parts[0]), int(parts[1]), int(parts[2])
            ts, te = parse_time(parts[3]), parse_time(parts[4])
            triples.append((h, r, t, ts, te))
            ents.update((h, t))
            rels.add(r)
            times.update((ts, te))
    return triples, ents, rels, times


def load_alignment(path):
    pairs = []
    with open(path, "r", encoding="utf-8") as f:
        next(f)
        for line in f:
            a, b = line.split("\t")[:2]
            pairs.append((int(a), int(b)))
    return pairs


triples1, e1_raw, r1_raw, t1_raw = load_quadruples(PAIR_DIR / f"quadruples_{lang_a}.tsv")
triples2, e2_raw, r2_raw, t2_raw = load_quadruples(PAIR_DIR / f"quadruples_{lang_b}.tsv")
all_pairs = load_alignment(PAIR_DIR / "alignment_all.tsv")

print(f"KG1 ({lang_a}): triples={len(triples1)} ents={len(e1_raw)} rels={len(r1_raw)} times={len(t1_raw)}")
print(f"KG2 ({lang_b}): triples={len(triples2)} ents={len(e2_raw)} rels={len(r2_raw)} times={len(t2_raw)}")
print(f"Alignment: {len(all_pairs)}")

KG1 (en): triples=40446 ents=12502 rels=24 times=618
KG2 (ja): triples=25789 ents=8351 rels=23 times=532
Alignment: 8317


In [6]:
# ============================================================
# 6. Remap IDs + TREA graph (begin/end + inverse relations)
# ============================================================

def make_maps(s):
    ids = sorted(s)
    return {x: i for i, x in enumerate(ids)}


e1_map, e2_map = make_maps(e1_raw), make_maps(e2_raw)
r_map = make_maps(r1_raw | r2_raw)
# time: reserve 0 as unknown; shift years to 1..T
all_times_raw = sorted(t1_raw | t2_raw)
time_map = {t: i + 1 for i, t in enumerate(all_times_raw)}
num_times = len(all_times_raw) + 1
sorted_time_ids = list(range(1, num_times))  # for sequential regularizer

N1, N2 = len(e1_map), len(e2_map)
NUM_ENTITIES = N1 + N2
NUM_RELS_RAW = len(r_map)


def remap_triples(triples, e_map, r_map, t_map, off=0):
    out = []
    for h, r, t, ts, te in triples:
        out.append((
            e_map[h] + off,
            r_map[r],
            e_map[t] + off,
            t_map.get(ts, 0),
            t_map.get(te, 0),
        ))
    return out


triples1_m = remap_triples(triples1, e1_map, r_map, time_map, 0)
triples2_m = remap_triples(triples2, e2_map, r_map, time_map, N1)
all_triples = triples1_m + triples2_m

valid = np.array([
    [e1_map[a], e2_map[b] + N1]
    for a, b in all_pairs if a in e1_map and b in e2_map
], dtype=np.int64)
np.random.shuffle(valid)

if len(valid) < TRAIN_SEED:
    raise ValueError(f"only {len(valid)} pairs < TRAIN_SEED={TRAIN_SEED}")

train_pair = valid[:TRAIN_SEED].copy()
dev_pair = valid[TRAIN_SEED:].copy()


def build_trea_graph(triples, num_rels):
    """Inward links: begin (h-r->t @ ts) and end (t-r_inv->h @ te)."""
    inward = defaultdict(list)
    for h, r, t, ts, te in triples:
        inward[t].append((h, r, ts))
        inward[h].append((t, r + num_rels, te))
    return inward


inward = build_trea_graph(all_triples, NUM_RELS_RAW)

print(f"N1={N1} N2={N2} entities={NUM_ENTITIES}")
print(f"rels_raw={NUM_RELS_RAW} times={num_times}")
print(f"train={len(train_pair)} dev={len(dev_pair)}")
print(f"inward non-empty: {sum(1 for e in range(NUM_ENTITIES) if inward[e])}")

N1=12502 N2=8351 entities=20853
rels_raw=24 times=620
train=200 dev=8117
inward non-empty: 20853


In [7]:
# ============================================================
# 7. Evaluation (CSLS)
# ============================================================

def csls_sim(Lvec, Rvec, k=10):
    Lvec = Lvec / np.linalg.norm(Lvec, axis=-1, keepdims=True).clip(1e-8)
    Rvec = Rvec / np.linalg.norm(Rvec, axis=-1, keepdims=True).clip(1e-8)
    sim = Lvec @ Rvec.T
    r_L = np.mean(np.sort(sim, axis=1)[:, -k:], axis=1)
    r_R = np.mean(np.sort(sim, axis=0)[-k:, :], axis=0)
    return -(2 * sim - r_L[:, None] - r_R[None, :])  # distance


def get_hits(vec, test_pair, top_k=(1, 5, 10), use_csls=True):
    Lvec = np.array([vec[e1] for e1, e2 in test_pair])
    Rvec = np.array([vec[e2] for e1, e2 in test_pair])
    if use_csls:
        sim_o = csls_sim(Lvec, Rvec, k=CSLS_K)
    else:
        Ln = Lvec / np.linalg.norm(Lvec, axis=-1, keepdims=True).clip(1e-8)
        Rn = Rvec / np.linalg.norm(Rvec, axis=-1, keepdims=True).clip(1e-8)
        sim_o = -Ln @ Rn.T

    order = sim_o.argsort(-1)
    top_lr = [0] * len(top_k); mrr = 0.0
    for i in range(len(test_pair)):
        rank = int(np.where(order[i] == i)[0][0])
        mrr += 1.0 / (rank + 1)
        for j, k in enumerate(top_k):
            if rank < k:
                top_lr[j] += 1
    print("Left->right:")
    for j, k in enumerate(top_k):
        print(f"  Hits@{k}: {top_lr[j] / len(test_pair) * 100:.2f}%")
    print(f"  MRR: {mrr / len(test_pair):.3f}")

print("Eval helpers ready.")

Eval helpers ready.


In [8]:
# ============================================================
# 8. TREA model: NAR + TRALayer
# ============================================================

class NAR(nn.Module):
    """Neighborhood Aggregation Representation (Eq. 1). Output 3*dim."""

    def __init__(self, ent_emb, rel_emb, time_emb, inward, num_entities):
        super().__init__()
        self.ent_emb = ent_emb
        self.rel_emb = rel_emb
        self.time_emb = time_emb
        self.num_entities = num_entities
        self._build_index(inward)

    def _build_index(self, inward):
        ent_rows, ent_cols = [], []
        rel_rows, rel_cols = [], []
        time_rows, time_cols = [], []
        for e in range(self.num_entities):
            links = inward.get(e, [])
            ent_rows.append(e); ent_cols.append(e)  # include self
            for src, r, t in links:
                ent_rows.append(e); ent_cols.append(src)
                rel_rows.append(e); rel_cols.append(r)
                time_rows.append(e); time_cols.append(t)
        self.register_buffer("ent_row", torch.tensor(ent_rows, dtype=torch.long))
        self.register_buffer("ent_col", torch.tensor(ent_cols, dtype=torch.long))
        self.register_buffer("rel_row", torch.tensor(rel_rows, dtype=torch.long))
        self.register_buffer("rel_col", torch.tensor(rel_cols, dtype=torch.long))
        self.register_buffer("time_row", torch.tensor(time_rows, dtype=torch.long))
        self.register_buffer("time_col", torch.tensor(time_cols, dtype=torch.long))

    @staticmethod
    def _scatter_mean(row, col, table, num_rows):
        out = torch.zeros(num_rows, table.shape[-1], device=table.device)
        out.index_add_(0, row, table[col])
        counts = torch.zeros(num_rows, device=table.device)
        counts.index_add_(0, row, torch.ones_like(row, dtype=torch.float32))
        return out / counts.clamp_min(1.0).unsqueeze(-1)

    def forward(self):
        ent_part = self._scatter_mean(self.ent_row, self.ent_col, self.ent_emb.weight, self.num_entities)
        rel_part = self._scatter_mean(self.rel_row, self.rel_col, self.rel_emb.weight, self.num_entities)
        time_part = self._scatter_mean(self.time_row, self.time_col, self.time_emb.weight, self.num_entities)
        return torch.cat([ent_part, rel_part, time_part], dim=-1)


class TRALayer(nn.Module):
    """
    Temporal Relational Attention (Eq. 2-4), matching trea_torch.py.

    Attention scores come only from relation/time embeddings (dim),
    while Wt/Wr scale the neighbor entity features (feat_dim = 3*dim).
    """

    def __init__(self, feat_dim, time_rel_dim):
        super().__init__()
        self.v_t = nn.Parameter(torch.empty(time_rel_dim))
        self.v_r = nn.Parameter(torch.empty(time_rel_dim))
        self.Wt = nn.Parameter(torch.empty(feat_dim))
        self.Wr = nn.Parameter(torch.empty(feat_dim))
        for p in (self.v_t, self.v_r, self.Wt, self.Wr):
            nn.init.normal_(p, std=0.1)

    @staticmethod
    def _segment_softmax(scores, seg_ids, num_segments):
        seg_max = torch.full((num_segments,), float("-inf"),
                             device=scores.device, dtype=scores.dtype)
        seg_max.scatter_reduce_(0, seg_ids, scores, reduce="amax", include_self=True)
        seg_max = torch.nan_to_num(seg_max, neginf=0.0)
        exp_s = torch.exp(scores - seg_max[seg_ids])
        seg_sum = torch.zeros(num_segments, device=scores.device, dtype=scores.dtype)
        seg_sum.scatter_add_(0, seg_ids, exp_s)
        return exp_s / seg_sum[seg_ids].clamp_min(1e-16)

    def forward(self, h, src_idx, dst_idx, rel_ids, time_ids, rel_emb, time_emb, num_entities):
        # Scores use RAW time/rel embeddings (width = dim), NOT entity features
        h_r = rel_emb[rel_ids]      # (E, dim)
        h_t = time_emb[time_ids]    # (E, dim)

        time_score = h_t @ self.v_t  # (E,)
        rel_score = h_r @ self.v_r   # (E,)

        alpha = self._segment_softmax(time_score, dst_idx, num_entities)
        beta = self._segment_softmax(rel_score, dst_idx, num_entities)

        src_feat = h[src_idx]  # (E, feat_dim)
        msg = (alpha.unsqueeze(-1) * (self.Wt * src_feat)
               + beta.unsqueeze(-1) * (self.Wr * src_feat))

        out = torch.zeros(num_entities, h.shape[-1], device=h.device, dtype=h.dtype)
        out.index_add_(0, dst_idx, msg)
        return torch.tanh(out)


class TREA(nn.Module):
    def __init__(self, num_entities, num_rels_doubled, num_times, dim, depth, inward):
        super().__init__()
        self.num_entities = num_entities
        self.ent_emb = nn.Embedding(num_entities, dim)
        self.rel_emb = nn.Embedding(num_rels_doubled, dim)
        self.time_emb = nn.Embedding(num_times, dim)
        nn.init.xavier_uniform_(self.ent_emb.weight)
        nn.init.xavier_uniform_(self.rel_emb.weight)
        nn.init.xavier_uniform_(self.time_emb.weight)

        self.nar = NAR(self.ent_emb, self.rel_emb, self.time_emb, inward, num_entities)
        feat_dim = 3 * dim
        self.layers = nn.ModuleList([TRALayer(feat_dim, dim) for _ in range(depth)])

        # flatten inward into tensors for TRA
        src_list, dst_list, rel_list, time_list = [], [], [], []
        for e in range(num_entities):
            for src, r, t in inward.get(e, []):
                src_list.append(src)
                dst_list.append(e)
                rel_list.append(r)
                time_list.append(t)
        if not src_list:
            src_list = [0]; dst_list = [0]; rel_list = [0]; time_list = [0]
        self.register_buffer("src_idx", torch.tensor(src_list, dtype=torch.long))
        self.register_buffer("dst_idx", torch.tensor(dst_list, dtype=torch.long))
        self.register_buffer("rel_ids", torch.tensor(rel_list, dtype=torch.long))
        self.register_buffer("time_ids", torch.tensor(time_list, dtype=torch.long))

    def forward(self):
        h0 = self.nar()
        outputs = [h0]
        h = h0
        for layer in self.layers:
            h = layer(h, self.src_idx, self.dst_idx, self.rel_ids, self.time_ids,
                      self.rel_emb.weight, self.time_emb.weight, self.num_entities)
            outputs.append(h)
        h_out = torch.cat(outputs, dim=-1)
        rel_mean = NAR._scatter_mean(self.dst_idx, self.rel_ids, self.rel_emb.weight, self.num_entities)
        time_mean = NAR._scatter_mean(self.dst_idx, self.time_ids, self.time_emb.weight, self.num_entities)
        return torch.cat([h_out, rel_mean, time_mean], dim=-1)


def mml_loss(emb, pos_pairs, gamma):
    """Margin multi-class log-loss with full negatives (Eq. 10)."""
    ei = emb[pos_pairs[:, 0]]
    ej = emb[pos_pairs[:, 1]]
    d_pos = ((ei - ej) ** 2).sum(-1)
    emb_sq = (emb ** 2).sum(-1)

    ei_sq = (ei ** 2).sum(-1, keepdim=True)
    d_all_j = ei_sq + emb_sq.unsqueeze(0) - 2 * (ei @ emb.t())
    term1 = torch.logsumexp(
        torch.cat([torch.zeros_like(d_pos).unsqueeze(1), gamma + d_pos.unsqueeze(1) - d_all_j], 1), 1
    )

    ej_sq = (ej ** 2).sum(-1, keepdim=True)
    d_all_i = ej_sq + emb_sq.unsqueeze(0) - 2 * (ej @ emb.t())
    term2 = torch.logsumexp(
        torch.cat([torch.zeros_like(d_pos).unsqueeze(1), gamma + d_pos.unsqueeze(1) - d_all_i], 1), 1
    )
    return (term1 + term2).mean()


def sequential_time_regularizer(time_emb, sorted_time_ids, h_tb):
    ids = torch.tensor(sorted_time_ids, dtype=torch.long, device=time_emb.device)
    if len(ids) < 2:
        return time_emb.sum() * 0.0
    h_seq = time_emb[ids]
    diffs = h_seq[1:] - h_seq[:-1] - h_tb
    return (diffs ** 2).sum(-1).mean()

print("TREA model defined.")

TREA model defined.


In [9]:
# ============================================================
# 9. Build model
# ============================================================

model = TREA(
    num_entities=NUM_ENTITIES,
    num_rels_doubled=NUM_RELS_RAW * 2,
    num_times=num_times,
    dim=DIM,
    depth=DEPTH,
    inward=inward,
).to(DEVICE)

h_tb = nn.Parameter(torch.zeros(DIM, device=DEVICE))
optimizer = torch.optim.RMSprop(list(model.parameters()) + [h_tb], lr=LR)
train_pair_t = torch.tensor(train_pair, dtype=torch.long, device=DEVICE)

print(model)
print("Params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

TREA(
  (ent_emb): Embedding(20853, 100)
  (rel_emb): Embedding(48, 100)
  (time_emb): Embedding(620, 100)
  (nar): NAR(
    (ent_emb): Embedding(20853, 100)
    (rel_emb): Embedding(48, 100)
    (time_emb): Embedding(620, 100)
  )
  (layers): ModuleList(
    (0-1): 2 x TRALayer()
  )
)
Params: 2153700


In [10]:
# ============================================================
# 10. Training
# ============================================================

history = []

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()

    emb = model()
    loss_p = mml_loss(emb, train_pair_t, GAMMA)
    loss_str = sequential_time_regularizer(model.time_emb.weight, sorted_time_ids, h_tb)
    loss = loss_p + LAMBDA_T * loss_str

    loss.backward()
    optimizer.step()

    history.append({
        "epoch": epoch + 1,
        "loss": float(loss.item()),
        "loss_mml": float(loss_p.item()),
        "loss_time": float(loss_str.item()),
    })

    if epoch % 10 == 9:
        model.eval()
        with torch.no_grad():
            emb_np = model().detach().cpu().numpy()
        print(f"[epoch {epoch+1:03d}] loss={loss.item():.4f} "
              f"(MML={loss_p.item():.4f}, STR={loss_str.item():.4f})")
        get_hits(emb_np, dev_pair, top_k=TOP_K, use_csls=USE_CSLS)

print("Training complete.")

[epoch 010] loss=3.1309 (MML=3.1272, STR=3.7712)
Left->right:
  Hits@1: 78.29%
  Hits@5: 88.12%
  Hits@10: 91.08%
  MRR: 0.829
[epoch 020] loss=2.5150 (MML=2.5118, STR=3.2791)
Left->right:
  Hits@1: 76.35%
  Hits@5: 86.92%
  Hits@10: 90.38%
  MRR: 0.814
[epoch 030] loss=2.3817 (MML=2.3788, STR=2.9602)
Left->right:
  Hits@1: 74.45%
  Hits@5: 85.81%
  Hits@10: 89.28%
  MRR: 0.799
[epoch 040] loss=2.3261 (MML=2.3233, STR=2.7337)
Left->right:
  Hits@1: 73.02%
  Hits@5: 85.12%
  Hits@10: 88.43%
  MRR: 0.786
[epoch 050] loss=2.2947 (MML=2.2922, STR=2.5592)
Left->right:
  Hits@1: 71.84%
  Hits@5: 84.13%
  Hits@10: 87.72%
  MRR: 0.775
[epoch 060] loss=2.2752 (MML=2.2728, STR=2.4187)
Left->right:
  Hits@1: 70.56%
  Hits@5: 83.22%
  Hits@10: 86.93%
  MRR: 0.764
[epoch 070] loss=2.2619 (MML=2.2596, STR=2.3012)
Left->right:
  Hits@1: 69.51%
  Hits@5: 82.63%
  Hits@10: 86.31%
  MRR: 0.756
[epoch 080] loss=2.2521 (MML=2.2499, STR=2.2010)
Left->right:
  Hits@1: 68.60%
  Hits@5: 81.80%
  Hits@10: 85.5

In [11]:
# ============================================================
# 11. Final evaluation + checkpoint
# ============================================================

model.eval()
with torch.no_grad():
    final_emb = model().detach().cpu().numpy()

print("Embedding shape:", final_emb.shape)
get_hits(final_emb, dev_pair, top_k=TOP_K, use_csls=USE_CSLS)

SAVE = f"/content/drive/MyDrive/trea_wikidata12k_{LANG_PAIR}_checkpoint.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "h_tb": h_tb.detach().cpu(),
    "config": {
        "LANG_PAIR": LANG_PAIR,
        "TRAIN_SEED": TRAIN_SEED,
        "DEPTH": DEPTH,
        "GAMMA": GAMMA,
        "LAMBDA_T": LAMBDA_T,
        "LR": LR,
        "DIM": DIM,
        "EPOCHS": EPOCHS,
        "N1": N1,
        "N2": N2,
    },
    "history": history,
}, SAVE)
print("Saved:", SAVE)

Embedding shape: (20853, 1100)
Left->right:
  Hits@1: 67.06%
  Hits@5: 80.42%
  Hits@10: 84.51%
  MRR: 0.733
Saved: /content/drive/MyDrive/trea_wikidata12k_en_ja_checkpoint.pt


## Notes

- Switch pair: set `LANG_PAIR` to `en_fr` / `en_fa` / `en_zh` / `en_ja`.
- Wikidata12K is much smaller than TKGL (~12k entities), so full-graph TREA fits on a standard Colab GPU.
- Keep the same `TRAIN_SEED` when comparing TREA to STEA / TEA-GNN / RREA.
- This is a paper-based reconstruction (no official TREA code was used).